# Import des librairies

In [19]:
import numpy as np
import heapq
from collections import defaultdict

# Lecture du fichier source

In [20]:
with open('real.txt') as fichier:
    matrix = np.array([
        list(line) 
        for line 
        in fichier.read().split('\n')
        ])
print(matrix)

[['#' '#' '#' ... '#' '#' '#']
 ['#' '.' '.' ... '#' 'E' '#']
 ['#' '.' '#' ... '#' '.' '#']
 ...
 ['#' '.' '#' ... '#' '.' '#']
 ['#' 'S' '.' ... '.' '.' '#']
 ['#' '#' '#' ... '#' '#' '#']]


# Initialisation

In [21]:
dim_matrix = len(matrix)

for num_line,line in enumerate(matrix): 
    for num_char,char in enumerate(line):
        if char == 'S':
            start = (num_line, num_char,1)
            matrix[num_line, num_char] = '.'
        if char == 'E':
            end = (num_line, num_char)
            matrix[num_line, num_char] = '.'
for line in matrix:
    print(line)

['#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#'
 '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#' '#']
['#' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '#' '.' '.' '.' '.' '.'
 '.' '.' '.' '.' '.' '.' '#' '.' '.' '.' '.' '.' '#' '.' '.' '.' '.' '.'
 '.' '.' '#' '.' '.' '.' '.' '.' '.' '.' '#' '.' '.' '.' '#' '.' '.' '.'
 '.' '.' '.' '.' '#' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.'
 '.' '.' '.' '.' '#' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.'
 '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '.' '

In [22]:
def func_in_grid(in_pos : tuple,
                 in_dim_matrix : int):
    if (0 <= in_pos[0] < dim_matrix) and (0 <= in_pos[1] < dim_matrix):
        return True
    else:
        return False

In [23]:
def func_recreate_path(in_precedents : dict,
                       in_final_pos : tuple,
                       in_start_pos):
    """
    A partir de la position d'arrivée, on remonte dans l'historique pour reconstruire le chemin
    """
    # Initialisation
    current_pos = in_final_pos
    file = [current_pos]
    visited = set()
    path = set()
    
    # Tant qu'on est pas arrivé à la position de départ
    while file:

        # On définit la nouvelle position
        current_pos = file.pop()

        # Si la position a deja été visitée
        if current_pos in visited:
            continue

        # Sinon, cela signifie que l'on va la traiter, donc qu'elle va être visitée
        visited.add(current_pos)
        path.add(current_pos[:2])

        # Si on arrive a la position de depart 
        if current_pos == in_start_pos:
            continue

        # On ajoute tous les prédecesseurs a la file d'attente
        for predecesseurs in in_precedents.get(current_pos,[]):
            file.append(predecesseurs)

    return path

# Traitement

In [24]:
# Dictionnaire contenant les mouvements possibles en fonction de la direction où l'on regarde (pas de marche arriere possible)
dict_moove = {
    0 : {
        3 : (0,-1),
        1 : (0,1),
        0 : (-1,0)
    },
    1 : {
        2 : (1,0),
        1 : (0,1),
        0 : (-1,0)
    },
    2 : {
        3 : (0,-1),
        2 : (1,0),
        1 : (0,1)
    },
    3 : {
        3 : (0,-1),
        2 : (1,0),
        0 : (-1,0)
    }
}

direction = 1

def func_moove_around(in_matrix : np.array,
                      in_start : tuple,
                      in_arrivee : tuple
                      ):
    
    distances = {}

    file_attente = [(0,in_start[0],in_start[1],1)] # (distance, x, y, direction)
    visited = set()
    precedents = defaultdict(list)
    distance_minimale = float('inf')
    path = set()

    while file_attente:
        # print()
        # print(file_attente)

        current_distance, num_row, num_col, direction = heapq.heappop(file_attente) 
        # print(f'current : {current_distance, num_row, num_col, direction}')
        pos = (num_row,num_col)
        pos_dir = (num_row, num_col, direction)

        # Si la position a deja été visitée
        if pos_dir in visited:
            continue
        # On va traiter la position, donc on la considere comme visitée
        visited.add(pos_dir)

        # On met a jour distances
        distances[pos_dir] = current_distance

        # Si on est arrivé 
        if pos == in_arrivee:

            # On définit la distance minimale
            distance_minimale = current_distance

            # On met à jours les tuiles visitées
            path |= func_recreate_path(
                    in_precedents = precedents,
                    in_start_pos = in_start,
                    in_final_pos = pos_dir
                    ) 
            continue

        # Pour chaque position voisine de notre position,direction
        for new_dir, delta in dict_moove[direction].items():

            # Potition du voisin
            pos_voisin = (pos[0] + delta[0], pos[1] + delta[1])

            # Si le voisin est hors grille, on passe au suivant
            if not func_in_grid(
                in_pos = pos_voisin,
                in_dim_matrix = dim_matrix
                ):
                continue
            
            # Si le voisin est un point
            if in_matrix[pos_voisin[0]][pos_voisin[1]] != '.':
                continue

            # Si le voisin a deja été visité, on passe au suivant
            if (pos_voisin[0], pos_voisin[1], new_dir) in visited:
                continue

            # 1 si on avance et 1001 si on tourne puis avance
            poid = 1 if direction == new_dir else 1001

            # On définit la nouvelle distance
            new_distance = current_distance + poid

            # Si la nouvelle distance dépasse la distance min deja trouvée, on arrete
            if new_distance > distance_minimale:
                 continue

            # Si on trouve une distance plus faible entre le point de départ et le voisin, on met a jour la distance 
            pos_dir_voisin = (pos_voisin[0], pos_voisin[1], new_dir)
            if new_distance <= distances.get(pos_dir_voisin,float('inf')) :
                # On met à jour la distance pour arriver au voisin
                distances[pos_dir_voisin] = new_distance
                # On sauvegarde prédecesseur
                precedents[pos_dir_voisin].append(pos_dir)
                # Ajout dans la file d'attente
                heapq.heappush(file_attente,(new_distance,pos_voisin[0],pos_voisin[1],new_dir))
           
    return path
            
tiles = func_moove_around(in_matrix = matrix,
                          in_start = start,
                          in_arrivee = end
                          )



In [25]:
len(tiles)

582

In [ ]:
from heapq import heappush, heappop
from collections import defaultdict

with open("real.txt") as fin:
    grid = fin.read().strip().split("\n")

n = len(grid)
for i in range(n):
    for j in range(n):
        if grid[i][j] == "S":
            start = (i, j)
        elif grid[i][j] == "E":
            end = (i, j)

dd = [[0, 1], [1, 0], [0, -1], [-1, 0]]

# Dijkstra's
q = [(0, 0, *start, 0, *start)]
cost = {}
deps = defaultdict(list)
while len(q) > 0:
    top = heappop(q)
    c, d, i, j, pd, pi, pj = top
    if (d, i, j) in cost:
        if cost[(d, i, j)] == c:
            deps[(d, i, j)].append((pd, pi, pj))
        continue

    never_seen_pos = True
    for newd in range(4):
        if (newd, i, j) in cost:
            never_seen_pos = True
            break
    if never_seen_pos:
        deps[(d, i, j)].append((pd, pi, pj))
    
    cost[(d, i, j)] = c


    if grid[i][j] == "#":
        continue

    if grid[i][j] == "E":
        end_dir = d
        print(c)
        break
    
    ii = i + dd[d][0]
    jj = j + dd[d][1]

    for nbr in [(c + 1, d, ii, jj, d, i, j),
                (c + 1000, (d + 1) % 4, i, j, d, i, j),
                (c + 1000, (d + 3) % 4, i, j, d, i, j)]:
        heappush(q, nbr)

# Go back through deps
stack = [(end_dir, *end)]
seen = set()
seen_pos = set()
while len(stack) > 0:
    top = stack.pop()
    if top in seen:
        continue
    seen.add(top)
    seen_pos.add(top[1:])

    for nbr in deps[top]:
        stack.append(nbr)

print(len(seen_pos))

# for i in range(n):
#     for j in range(n):
#         if (i, j) in seen:
#             print("OO", end="")
#         else:
#             print("..", end="")
#     print()

105496
524
